# Iceberg INSERT Diagnostic

**Problem:** INSERT hangs at `[Stage 0:>` even for `testdb`.

Run each section in order in a **fresh kernel**. Stop when one INSERT succeeds.

Share outputs from **Diagnostics** and whichever INSERT step you reach.

## A0 — Stop any existing Spark session (run first on fresh kernel)

In [ ]:
try:
    spark.stop()
    print("Stopped existing spark session")
except Exception as e:
    print("No spark session to stop:", e)

## A1 — SparkSession: EXACT Quickstart config (single filesystem only)

Copied verbatim from `Iceberg_PySpark_Quickstart_ADLS.ipynb`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
  .appName("1.1 - Ingest") \
  .config("spark.hadoop.fs.s3a.s3guard.ddb.region", "us-east-2")\
  .config("spark.yarn.access.hadoopFileSystems", "abfs://data@go01demoazure.dfs.core.windows.net/go01-az-dl")\
  .config("spark.jars","/opt/spark/optional-lib/iceberg-spark-runtime-3.5_2.12-1.5.2.1.25.731.0-41.jar") \
  .config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.spark_catalog","org.apache.iceberg.spark.SparkSessionCatalog") \
  .config("spark.sql.catalog.local","org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type","hadoop") \
  .config("spark.sql.catalog.spark_catalog.type","hive") \
  .getOrCreate()

print("Spark version:", spark.version)

## A2 — Diagnostics (share this output)

In [ ]:
sc = spark.sparkContext
print("master:", sc.master)
print("appId:", sc.applicationId)
print("deployMode:", spark.conf.get("spark.submit.deployMode", "(not set)"))
print("spark.jars:", spark.conf.get("spark.jars", ""))
print("filesystems:", spark.conf.get("spark.yarn.access.hadoopFileSystems", ""))
print("dynamicAllocation:", spark.conf.get("spark.dynamicAllocation.enabled", ""))
print("executor.instances:", spark.conf.get("spark.executor.instances", ""))
print("executor.memory:", spark.conf.get("spark.executor.memory", ""))
print("k8s.namespace:", spark.conf.get("spark.kubernetes.namespace", ""))
print("k8s.authenticate.driver.serviceAccountName:", spark.conf.get("spark.kubernetes.authenticate.driver.serviceAccountName", ""))

## B1 — Quickstart INSERT test (exact notebook cells)

If this hangs, the issue is environmental (YARN/resources), not `airline_irop`.

In [ ]:
import time

spark.sql("USE spark_catalog.testdb")
spark.sql("SHOW CURRENT NAMESPACE").show()
spark.sql("CREATE TABLE IF NOT EXISTS newtesttable (id bigint, data string) USING iceberg")

print("Starting INSERT at", time.strftime("%H:%M:%S"))
t0 = time.time()
spark.sql("INSERT INTO spark_catalog.testdb.newtesttable VALUES (1, 'x'), (2, 'y'), (3, 'z')")
print(f"INSERT finished in {time.time()-t0:.1f}s")
spark.sql("SELECT * FROM spark_catalog.testdb.newtesttable").show()

## B2 — Retry with fixed executors (if B1 hung)

Your A2 shows `dynamicAllocation: true` with no fixed `executor.instances` — INSERT waits for K8s to spin up executor pods. This cell disables dynamic allocation and requests 1 executor.

Run **A0** again, then this cell instead of A1+B1.

In [ ]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder\
  .appName("IROP Insert Fixed Executors") \
  .config("spark.hadoop.fs.s3a.s3guard.ddb.region", "us-east-2")\
  .config("spark.yarn.access.hadoopFileSystems", "abfs://data@go01demoazure.dfs.core.windows.net/go01-az-dl")\
  .config("spark.jars","/opt/spark/optional-lib/iceberg-spark-runtime-3.5_2.12-1.5.2.1.25.731.0-41.jar") \
  .config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.spark_catalog","org.apache.iceberg.spark.SparkSessionCatalog") \
  .config("spark.sql.catalog.local","org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type","hadoop") \
  .config("spark.sql.catalog.spark_catalog.type","hive") \
  .config("spark.dynamicAllocation.enabled", "false")\
  .config("spark.executor.instances", "2")\
  .config("spark.executor.memory", "4g")\
  .config("spark.executor.cores", "2")\
  .config("spark.kubernetes.executor.request.cores", "1")\
  .config("spark.kubernetes.executor.limit.cores", "2")\
  .getOrCreate()

print("master:", spark.sparkContext.master)
print("executor.instances:", spark.conf.get("spark.executor.instances"))

spark.sql("USE spark_catalog.testdb")
spark.sql("CREATE TABLE IF NOT EXISTS newtesttable (id bigint, data string) USING iceberg")
print("Starting INSERT at", time.strftime("%H:%M:%S"))
t0 = time.time()
spark.sql("INSERT INTO spark_catalog.testdb.newtesttable VALUES (99, 'fixed-exec')")
print(f"INSERT finished in {time.time()-t0:.1f}s")

## C1 — airline_irop (only after B1 or B2 INSERT works)

Uses explicit external hive LOCATION (same root path as testdb snapshots in Quickstart).

In [ ]:
import time

ADLS = "abfs://data@go01demoazure.dfs.core.windows.net"
DB_LOC = f"{ADLS}/warehouse/tablespace/external/hive/airline_irop.db"
TABLE = "flight_operational_events_smoke"
TABLE_LOC = f"{DB_LOC}/{TABLE}"
FULL = f"spark_catalog.airline_irop.{TABLE}"

spark.sql(f"CREATE DATABASE IF NOT EXISTS airline_irop LOCATION '{DB_LOC}'")
spark.sql("USE spark_catalog.airline_irop")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
spark.sql(f"""
  CREATE TABLE {TABLE} (
    event_id STRING, pnr STRING, customer_id STRING,
    flight_number STRING, itinerary STRING,
    orig_connection_mins INT, new_connection_mins INT,
    misconnect_risk BOOLEAN, event_timestamp TIMESTAMP
  ) USING iceberg LOCATION '{TABLE_LOC}'
""")

t0 = time.time()
spark.sql(f"""
  INSERT INTO {FULL} VALUES (
    'EVT-SMOKE-1','PNR-TEST','CUST-000','EK002','LHR-DXB-SIN',
    90,35,TRUE,timestamp('2026-08-12 11:30:00')
  )
""")
print(f"airline_irop INSERT finished in {time.time()-t0:.1f}s")
spark.sql(f"SELECT * FROM {FULL}").show()

---
### What to share
1. **A2** full diagnostics output
2. Does **B1** finish or hang? How long did you wait?
3. If B1 hung, does **B2** (fixed executors) finish?
4. Does the **original** `Iceberg_PySpark_Quickstart_ADLS.ipynb` INSERT cell still work in this same session/kernel?